# 07. Python Tuples (5+ Years Interview Guide)
Comprehensive analysis of tuple immutability, memory compactness, singleton comma gotcha, extended unpacking, namedtuple, and hashability protocols.

### Key 5-Year Interview Concepts Covered:
- **Tuple Memory Optimization**: Why tuples consume less memory and have lower allocation overhead than lists (no over-allocation, fixed struct caching).
- **Singleton Comma Gotcha**: Why `(42)` is an `int` while `(42,)` is a `tuple`.
- **Extended Star Unpacking**: Unpacking dynamic head/tail slices `first, *middle, last = seq`.
- **Shallow vs Deep Immutability**: Why a tuple containing a list cannot be hashed or used as a dictionary key.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [1]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

Using CSV file path: data/raw_transactions.csv


### 1. Tuple Creation & Memory Architecture
**Explanation**: Tuples (`PyTupleObject`) are ordered, immutable sequences. Because tuples cannot change size after allocation, CPython allocates the exact memory required without over-allocation padding. Furthermore, CPython maintains a free list of small tuples to recycle deallocated memory, making tuple creation faster than list creation.

**Syntax**: `point = (x, y)` / `empty_tuple = ()`

In [2]:
transaction_tuple = (100.0, 200.0)
print(transaction_tuple)

(100.0, 200.0)


### 2. Singleton Comma Gotcha (`(val,)`)
**Explanation**: In Python, parentheses `()` are used for grouping expressions as well as tuple syntax. Therefore, `(50)` is evaluated as an integer `50`. To define a single-element tuple (singleton), a trailing comma is syntactically mandatory: `(50,)`. Forgetting this trailing comma is one of the most common junior-level bugs in SQL query parameter tuples.

**Syntax**: `single_item_tuple = (value,)`

In [3]:
singleton_comma_tuple = (100.0,)
print(type(singleton_comma_tuple))

<class 'tuple'>


### 3. Tuple Indexing & O(1) Access
**Explanation**: Tuple elements are accessed by 0-based index in O(1) constant time, identical to lists. Tuples support negative indexing (`tuple[-1]`) relative to the end of the sequence.

**Syntax**: `elem = my_tuple[index]  # O(1) access`

In [4]:
transaction_tuple = (100.0, 200.0)
print(transaction_tuple[0])

100.0


### 4. Slicing Tuples (`[start:stop:step]`)
**Explanation**: Slicing a tuple `my_tuple[1:4]` returns a new tuple containing the sliced element pointers in O(K) time. Slicing the full tuple `my_tuple[:]` returns the exact same tuple object (same `id()`) because tuples are immutable, unlike lists where `[:]` creates a new shallow copy.

**Syntax**: `sub_tuple = my_tuple[start:stop:step]`

In [5]:
transaction_tuple = (10.0, 20.0, 30.0)
print(transaction_tuple[1:3])

(20.0, 30.0)


### 5. Immutability Verification
**Explanation**: Attempting to modify, reassign, or delete a tuple element (`t[0] = 'new'`) raises a `TypeError: 'tuple' object does not support item assignment`. This guarantees data integrity, making tuples safe for multi-threaded access without locks and suitable for dictionary keys (if all elements are hashable).

**Syntax**: `t[0] = val  # Raises TypeError`

In [6]:
transaction_tuple = (100.0, 200.0)
try:
    transaction_tuple[0] = 500.0
except TypeError as error_message:
    print('TypeError caught:', error_message)

TypeError caught: 'tuple' object does not support item assignment


### 6. Shallow vs Deep Immutability & Hashability
**Explanation**: A tuple's container bindings are immutable (it will always point to the same object IDs), but if the tuple contains a mutable object (e.g. a list `t = (1, [2, 3])`), the contents of the inner list CAN be modified in-place! Furthermore, a tuple is hashable `hash(t)` ONLY if all elements inside it are hashable. A tuple containing a list cannot be hashed or stored in a set/dict.

**Syntax**: `hash((1, 2, 'a'))  # Valid` / `hash((1, [2]))  # TypeError: unhashable`

In [7]:
tuple_with_nested_list = (1, [])
tuple_with_nested_list[1].append(2)
print('Modified nested list inside tuple:', tuple_with_nested_list)

Modified nested list inside tuple: (1, [2])


### 7. Tuple Packing
**Explanation**: When multiple comma-separated values are assigned to a single variable without parentheses, Python automatically packs them into a tuple: `point = 10, 20`. This is the fundamental mechanism behind Python's multiple return values from functions.

**Syntax**: `packed_tuple = val1, val2, val3`

In [8]:
unpacked_transaction = 1, 'Visa', 2.0
print(unpacked_transaction)

(1, 'Visa', 2.0)


### 8. Tuple Unpacking (Destructuring)
**Explanation**: Tuple unpacking assigns elements from a tuple into separate target variables in a single step: `x, y = (10, 20)`. The number of variables on the left must exactly match the number of elements in the tuple, otherwise a `ValueError: too many values to unpack` or `not enough values to unpack` is raised.

**Syntax**: `var1, var2 = tuple_of_two`

In [9]:
first_coordinate, second_coordinate = (10, 20)
print(first_coordinate, second_coordinate)

10 20


### 9. Extended Star Unpacking (`*rest`)
**Explanation**: Extended unpacking (PEP 3132) uses the star operator `*` to capture variable-length sub-sequences into a list: `first, *middle, last = sequence`. Only one starred expression is permitted per assignment target, and it captures 0 or more elements cleanly.

**Syntax**: `first, *rest = sequence` / `*head, tail = sequence`

In [10]:
first_item, *remaining_items = (1, 2, 3)
print('First:', first_item, 'Rest:', remaining_items)

First: 1 Rest: [2, 3]


### 10. Lightweight Data Structures with `namedtuple`
**Explanation**: `collections.namedtuple` generates lightweight, memory-efficient subclass tuples with named fields accessible by name attribute (`rec.amount`) as well as positional index (`rec[0]`). Unlike full custom classes, `namedtuple` instances have no `__dict__` overhead, providing the memory efficiency of tuples with the readability of classes.

**Syntax**: `from collections import namedtuple; Record = namedtuple('Record', ['id', 'amt'])`

In [11]:
from collections import namedtuple
NamedTransactionTuple = namedtuple('NamedTransactionTuple', ['id', 'amt'])
transaction_record = NamedTransactionTuple('TX1', 10.0)
print(transaction_record.id, transaction_record.amt)

TX1 10.0


### 11. Tuple Concatenation & Replication (`+` and `*`)
**Explanation**: The `+` operator combines tuples into a new tuple object, and `*` replicates the tuple N times. Because tuples are immutable, both operations allocate a new tuple instance in memory.

**Syntax**: `combined = tuple_a + tuple_b` / `repeated = tuple_a * 3`

In [12]:
print((1, 2) + (3,) * 2)

(1, 2, 3, 3)


### 12. Tuple Searching Methods (`count()` and `index()`)
**Explanation**: Tuples provide only two public methods: `.index(value)` (returns the index of the first occurrence, O(N)) and `.count(value)` (counts occurrences, O(N)). The lack of mutation methods (`append`, `pop`, `insert`) keeps the tuple class minimal and lightweight.

**Syntax**: `idx = t.index(val)` / `cnt = t.count(val)`

In [13]:
target_tuple = (1, 2, 2)
print('Index:', target_tuple.index(2), 'Count:', target_tuple.count(2))

Index: 1 Count: 2


### 13. Membership Testing on Tuples (`in`)
**Explanation**: Membership checking `item in my_tuple` scans the tuple linearly in O(N) time. For static lookup sets in code (e.g. `if status in ('ACTIVE', 'PENDING'):`), CPython's peephole optimizer folds constant tuples into bytecode constants for fast evaluation.

**Syntax**: `if item in target_tuple: ...`

In [14]:
target_tuple = (1, 2)
print(1 in target_tuple)

True


### 14. Returning Multiple Values from Functions
**Explanation**: When a function returns multiple comma-separated values `return a, b, c`, Python packs them into a single tuple. Callers can unpack them directly: `x, y, z = get_coordinates()`. This is both clean and idiomatic.

**Syntax**: `def get_user(): return user_id, user_name`

In [15]:
def calculate_limits(): return 100.0, 200.0
print(calculate_limits())

(100.0, 200.0)


### 15. Sequence Conversion to Tuple (`tuple()`)
**Explanation**: The `tuple(iterable)` constructor converts any iterable into an immutable tuple. If the input is already a tuple, `tuple(t)` returns the exact same object `id(t)` in O(1) time without copying.

**Syntax**: `t = tuple(list_or_generator)`

In [16]:
print(tuple('abc'))

('a', 'b', 'c')


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Structured data parsing, memory-efficient record modeling with `namedtuple`, and immutable tuple slicing on transaction logs.


In [17]:
# Solution:
with open(csv_path, 'r') as f:
    f.readline()
    row = tuple(f.readline().strip().split(','))
    tx_id, cust_id, merch_id, *rest = row
    print('Tx ID:', tx_id, '| Remaining data fields tuple:', rest[:3])


Tx ID: TX110686 | Remaining data fields tuple: ['1216.33', 'Visa', 'Failed']


### Q2: Structured Transaction Processing with `namedtuple`
**Explanation**: **Scenario**: Parse raw transaction CSV rows into strongly typed `namedtuple` instances (`TransactionRecord`), compute summary metrics, and demonstrate field-level dot access.

**Syntax**: `TransactionRecord = namedtuple('TransactionRecord', ['id', 'amount', 'region'])`

In [18]:
# Solution:
from collections import namedtuple
TxLog = namedtuple('TxLog', ['tx_id', 'region', 'card'])
logs = []
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(5):
        row = f.readline().strip().split(',')
        logs.append(TxLog(row[0], row[9], row[4]))
        
for log in logs:
    print(f'ID: {log.tx_id} | Reg: {log.region} | Card: {log.card}')


ID: TX110686 | Reg: North | Card: Visa
ID: TX107170 | Reg: East | Card: MasterCard
ID: TX108328 | Reg: East | Card: Discover
ID: TX108563 | Reg: West | Card: Amex
ID: TX107002 | Reg: East | Card: MasterCard
